In [1]:
using SpeedyWeather, CairoMakie, GLMakie

In [2]:
spectral_grid = SpectralGrid()

SpectralGrid{Spectrum{...}, OctahedralGaussianGrid{...}}
├ Number format: Float32
├ Spectral:      T31 LowerTriangularMatrix
├ Grid:          48-ring OctahedralGaussianGrid, 3168 grid points
├ Resolution:    3.61°, 401km (at 6371km radius)
├ Vertical:      8-layer atmosphere, 2-layer land
└ Architecture:  CPU using Array

In [3]:
model = PrimitiveWetModel(spectral_grid)
simulation = initialize!(model)

Simulation{PrimitiveWetModel}
├ prognostic_variables::PrognosticVariables{...}
├ diagnostic_variables::DiagnosticVariables{...}
└ model::PrimitiveWetModel{...}

## Output variables

In [4]:
model.output

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ v: meridional wind [m/s]
 ├ humid: specific humidity [kg/kg]
 ├ temp: temperature [degC]
 ├ u: zonal wind [m/s]
 ├ mslp: mean sea-level pressure [hPa]
 └ vor: relative vorticity [s^-1]

In [5]:
add!(model, SpeedyWeather.RadiationOutput()...)

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ sru: Surface shortwave radiation up [W/m^2]
 ├ temp: temperature [degC]
 ├ srd: Surface shortwave radiation down [W/m^2]
 ├ mslp: mean sea-level pressure [hPa]
 ├ vor: relative vorticity [s^-1]
 ├ osr: Outgoing shortwave radiation [W/m^2]
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 ├ albedo: albedo [1]
 ├ lrd: Surface longwave radiation down [W/m^2]
 ├ humid: specific humidity [kg/kg]
 ├ olr: Outgoing longwave radiation [W/m^2]
 └ lru: Surfa

In [6]:
add!(model, SpeedyWeather.SurfaceFluxesOutput()...)

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ slf: Surface latent heat flux (positive up) [W/m^2]
 ├ sru: Surface shortwave radiation up [W/m^2]
 ├ temp: temperature [degC]
 ├ srd: Surface shortwave radiation down [W/m^2]
 ├ mslp: mean sea-level pressure [hPa]
 ├ vor: relative vorticity [s^-1]
 ├ osr: Outgoing shortwave radiation [W/m^2]
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 ├ albedo: albedo [1]
 ├ lrd: Surface longwave radiation down [W/m^2]
 ├ shf: Surface sensible heat flux (po

In [23]:
simulation.diagnostic_variables.physics.sensible_heat_flux

3168-element, 48-ring OctahedralGaussianField{Float32, 1} as Array on CPU
 46.88108f0
 39.860634f0
 31.37547f0
 22.205515f0
 14.143892f0
  8.885732f0
  6.2358828f0
  4.863484f0
  3.9429836f0
  3.5111647f0
  ⋮
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0

In [ ]:
simulation.diagnostic_variables.physics.surface_latent_heat_flux

3168-element, 48-ring OctahedralGaussianField{Float32, 1} as Array on CPU
 22.441494f0
 19.629692f0
 15.677593f0
 11.134917f0
  7.073079f0
  4.420079f0
  3.0803332f0
  2.3774915f0
  1.8975257f0
  1.6555068f0
  ⋮
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0

In [27]:
# run!(simulation, period=Day(30))
run!(simulation, period=Day(90))

In [28]:
heatmap(simulation.diagnostic_variables.physics.sensible_heat_flux)

In [29]:
simulation.prognostic_variables.clock

Clock
├ time::DateTime = 2000-04-30T00:00:00
├ start::DateTime = 2000-01-31T00:00:00
├ period::Second = 7776000 seconds
├ timestep_counter::Int64 = 3240
├ n_timesteps::Int64 = 3240
└ Δt::Millisecond = 2400000 milliseconds

In [32]:
function calc_global_mean(field, model)
    a00 = real(transform(field)[1])
    return a00 / model.spectral_transform.norm_sphere    
end
# mean_per_m2 = a00 / model.spectral_transform.norm_sphere

calc_global_mean (generic function with 1 method)

In [ ]:
mean_SHF = calc_global_mean(simulation.diagnostic_variables.physics.sensible_heat_flux, model)

32.134113f0

In [38]:
mean_LHF = calc_global_mean(simulation.diagnostic_variables.physics.surface_latent_heat_flux, model)

42.10491f0

In [42]:
simulation.diagnostic_variables.physics

PhysicsVariables
├ grid: OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}
├ ocean: DynamicsVariablesOcean{Float32, Array, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Field{Float32, 1, Vector{Float32}, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ land: DynamicsVariablesLand{Float32, Array, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Field{Float32, 1, Vector{Float32}, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ rain_large_scale: 3168-element, 48-ring Field{Float32}
├ rain_convection: 3168-element, 48-ring Field{Float32}
├ snow_large_scale: 3168-element, 48-ring Field{Float32}
├ snow_convection: 3168-element, 48-ring Field{Float32}
├ total_precipitation_rate: 3168-element, 48-ring Field{Float32}
├ cloud_top: 3168-element, 48-ring Field{Float32}

In [ ]:
function calc_trenberth_variables(simulation, model, MeanSumFlag)
    # this is a function to calculate the variables we need to plot the Trenberth diagram.
    # simulation -> the SpeedyWeather simulation output.
    # model -> the SpeedyWeather model structure.
    # MeanSumFlag -> 0 for clculating using the area mean of the fluxes [W/m^2] and 1 for calculating for the global sum [W]. 

    # identify the relevat fields:
    LHF = simulation.diagnostic_variables.physics.surface_latent_heat_flux # surface latent heat up
    SHF = simulation.diagnostic_variables.physics.sensible_heat_flux # surface sensible heat up
    SSRU = simulation.diagnostic_variables.physics.surface_shortwave_up # surface shortwave up
    SLRU = simulation.diagnostic_variables.physics.surface_longwave_up # surface longwave up
    SSRD = simulation.diagnostic_variables.physics.surface_shortwave_down # surface shortwave down
    SLRD = simulation.diagnostic_variables.physics.surface_longwave_down # surface longwave down
    OSR = simulation.diagnostic_variables.physics.outgoing_shortwave_radiation # outgoing shortwave radiation (TOA)
    OLR = simulation.diagnostic_variables.physics.outgoing_longwave_radiation # outgoing longwave radiation (TOA)
    albedo = simulation.diagnostic_variables.physics.albedo # albedo

    # calculting tht global mean/sum
    if MeanSumFlag == 0

    elseif MeanSumFlag == 1

    else
        error('not working for this Flag')
    end

end


calc_trenberth_variables (generic function with 1 method)

In [35]:
function calc_global_sum(field, model)
    a00 = real(transform(field)[1])
    mean_per_m2 = a00 / model.spectral_transform.norm_sphere
    total_W = mean_per_m2 * (4*pi*model.planet.radius^2)
    return total_W
end


calc_global_sum (generic function with 1 method)